In [1]:
import nbloss

print("Imported from:", nbloss.__file__)

Imported from: C:\Users\koeng\Downloads\nbloss-main\nbloss-main\nbloss\__init__.py


# Step 1 — Load and inspect the Framingham dataset

# Framingham Data Preparation

This script constructs the dataset used for the 15-year Framingham ANYCHD prediction experiment.

Starting from the original longitudinal Framingham dataset, it:

- Retains the baseline examination (`PERIOD == 1`).
- Excludes participants with prevalent coronary heart disease at baseline.
- Defines incident ANYCHD within 15 years as the binary outcome.
- Excludes participants whose 15-year outcome cannot be determined because of insufficient follow-up.
- Selects the baseline predictors used in the modelling experiments.
- Creates the final modelling dataset without participant identifiers.

The resulting dataset is used to compare conventional NLL/BCE training with Smooth Net Benefit training. 

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# Step 1 — Load and prepare Framingham data
# ============================================================

DATA_PATH = Path("Data") / "framingham.csv"

df_raw = pd.read_csv(
    DATA_PATH,
    sep=None,
    engine="python",
)

df_raw = df_raw.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
)


# ------------------------------------------------------------
# Keep baseline examination only
# ------------------------------------------------------------

df = df_raw.loc[
    df_raw["PERIOD"] == 1
].copy()

assert df["RANDID"].is_unique


# ------------------------------------------------------------
# Exclude prevalent coronary heart disease
# ------------------------------------------------------------

df = df.loc[
    df["PREVCHD"] == 0
].copy()

assert len(df) == 4240


# ------------------------------------------------------------
# Construct 15-year incident ANYCHD outcome
# ------------------------------------------------------------

HORIZON_YEARS = 15
HORIZON_DAYS = 365.25 * HORIZON_YEARS

event_15y = (
    (df["ANYCHD"] == 1)
    & (df["TIMECHD"] <= HORIZON_DAYS)
)

non_event_15y = (
    ~event_15y
    & (df["TIMECHD"] >= HORIZON_DAYS)
)

known_outcome = (
    event_15y
    | non_event_15y
)

df = df.loc[
    known_outcome
].copy()

df["FifteenYearCHD"] = (
    event_15y.loc[df.index]
    .astype(int)
)


# ------------------------------------------------------------
# Retain modelling variables
# ------------------------------------------------------------

PREDICTOR_COLS = [
    "SEX",
    "TOTCHOL",
    "AGE",
    "SYSBP",
    "DIABP",
    "CURSMOKE",
    "CIGPDAY",
    "BMI",
    "DIABETES",
    "BPMEDS",
    "HEARTRTE",
    "GLUCOSE",
    "educ",
    "PREVSTRK",
    "PREVHYP",
]

TARGET_COL = "FifteenYearCHD"

model_df = df[
    PREDICTOR_COLS + [TARGET_COL]
].copy()


# ------------------------------------------------------------
# Ensure modelling variables are numeric
# ------------------------------------------------------------

for col in model_df.columns:
    model_df[col] = pd.to_numeric(
        model_df[col]
        .astype(str)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert model_df[TARGET_COL].isin([0, 1]).all()

print("Modelling dataset shape:", model_df.shape)
print(
    "15-year ANYCHD prevalence:",
    f"{model_df[TARGET_COL].mean():.3%}",
)

print("\nMissing values:")
print(
    model_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

# Step 2 — Create repeated train/test splits and preprocess the data

This step creates the five rotating 80/20 train/test splits used for the Framingham logistic-regression experiment. All preprocessing steps are fitted on the training data only and then applied to the corresponding test fold. No validation set is used. Two preprocessing variants are available: `mild`, where education is retained as a single ordinal variable, and `moderate`, where education is one-hot encoded.

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch

from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler


# ============================================================
# Step 2 — Framingham XGBoost preprocessing
# New 15-year incident ANYCHD analysis
# ============================================================


@dataclass
class SplitPack:
    run_id: int
    seed: int
    feature_cols: list

    X_train: np.ndarray
    X_test: np.ndarray
    y_train: np.ndarray
    y_test: np.ndarray

    X_train_t: torch.Tensor
    X_test_t: torch.Tensor
    y_train_t: torch.Tensor
    y_test_t: torch.Tensor

    imputer_bin: SimpleImputer
    imputer_cont: SimpleImputer
    scaler_cont: StandardScaler

    binary_cols: list
    categorical_cols: list
    continuous_cols: list
    engineered_cont_cols: list
    ohe_cols: list

    y_train_mean: float
    y_test_mean: float

    fe_level: str = "mild"


# ============================================================
# Variable definitions
# ============================================================

BINARY_COLS = [
    "SEX",
    "CURSMOKE",
    "DIABETES",
    "BPMEDS",
    "PREVSTRK",
    "PREVHYP",
]

CATEGORICAL_COLS = [
    "educ",
]

CONTINUOUS_COLS = [
    "AGE",
    "CIGPDAY",
    "TOTCHOL",
    "SYSBP",
    "DIABP",
    "BMI",
    "HEARTRTE",
    "GLUCOSE",
]


# ============================================================
# Validation
# ============================================================

def _validate_columns(
    df: pd.DataFrame,
    target_col: str,
):
    required = set(
        BINARY_COLS
        + CATEGORICAL_COLS
        + CONTINUOUS_COLS
        + [target_col]
    )

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )


# ============================================================
# Prepare base dataframe
# ============================================================

def _prepare_base_frame(
    df: pd.DataFrame,
    target_col: str,
) -> pd.DataFrame:

    cols = (
        BINARY_COLS
        + CATEGORICAL_COLS
        + CONTINUOUS_COLS
        + [target_col]
    )

    out = df[cols].copy()

    return out


# ============================================================
# Optional feature engineering
#
# Currently deliberately empty.
# This preserves the structure of the previous pipeline
# without adding arbitrary new predictors.
# ============================================================

def _add_engineered_features_after_imputation(
    X_cont_df: pd.DataFrame,
    X_bin_df: pd.DataFrame,
    fe_level: str,
):

    if fe_level not in (
        "mild",
        "moderate",
    ):
        raise ValueError(
            "fe_level must be 'mild' or 'moderate'"
        )

    X_eng = pd.DataFrame(
        index=X_cont_df.index
    )

    engineered_cols = []

    return X_eng, engineered_cols


# ============================================================
# Education:
# no OHE
# ============================================================

def _education_no_ohe_train_test(
    X_train_cat: pd.DataFrame,
    X_test_cat: pd.DataFrame,
):

    imputer_edu = SimpleImputer(
        strategy="most_frequent"
    )

    X_train_edu = pd.DataFrame(
        imputer_edu.fit_transform(
            X_train_cat[["educ"]]
        ),
        columns=["educ"],
        index=X_train_cat.index,
    ).astype(float)

    X_test_edu = pd.DataFrame(
        imputer_edu.transform(
            X_test_cat[["educ"]]
        ),
        columns=["educ"],
        index=X_test_cat.index,
    ).astype(float)

    return (
        X_train_edu,
        X_test_edu,
        ["educ"],
        imputer_edu,
    )


# ============================================================
# Education:
# one-hot encoding
# ============================================================

def _ohe_education_train_test(
    X_train_cat: pd.DataFrame,
    X_test_cat: pd.DataFrame,
):

    X_train_cat = X_train_cat.copy()
    X_test_cat = X_test_cat.copy()

    for frame in (
        X_train_cat,
        X_test_cat,
    ):
        frame["educ"] = (
            frame["educ"]
            .astype("object")
            .where(
                ~frame["educ"].isna(),
                "missing",
            )
        )

        frame["educ"] = (
            frame["educ"]
            .astype(str)
        )

    train_ohe = pd.get_dummies(
        X_train_cat,
        columns=["educ"],
        prefix="educ",
        drop_first=False,
    )

    test_ohe = pd.get_dummies(
        X_test_cat,
        columns=["educ"],
        prefix="educ",
        drop_first=False,
    )

    # Only categories observed in training define the feature space
    ohe_cols = train_ohe.columns.tolist()

    test_ohe = test_ohe.reindex(
        columns=ohe_cols,
        fill_value=0.0,
    )

    train_ohe = train_ohe.astype(float)
    test_ohe = test_ohe.astype(float)

    return (
        train_ohe,
        test_ohe,
        ohe_cols,
    )


# ============================================================
# Create 5 rotating stratified folds
# ============================================================

def make_framingham_splits_5x(
    df: pd.DataFrame,
    *,
    target_col: str = "FifteenYearCHD",
    base_seed: int = 42,
    n_runs: int = 5,
    device: str = "cpu",
    fe_level: str = "mild",
):

    if fe_level not in (
        "mild",
        "moderate",
    ):
        raise ValueError(
            "fe_level must be 'mild' or 'moderate'"
        )

    if n_runs != 5:
        raise ValueError(
            "This regime uses exactly 5 rotating folds, "
            "so n_runs must be 5."
        )

    # --------------------------------------------------------
    # Validate input
    # --------------------------------------------------------

    _validate_columns(
        df,
        target_col,
    )

    df_base = _prepare_base_frame(
        df,
        target_col,
    )

    # --------------------------------------------------------
    # Outcome
    # --------------------------------------------------------

    y_all = (
        df_base[target_col]
        .to_numpy(dtype=int)
    )

    if not np.isin(
        y_all,
        [0, 1],
    ).all():
        raise ValueError(
            "Target must contain only 0 and 1."
        )

    X_df = (
        df_base
        .drop(columns=[target_col])
        .copy()
    )

    # --------------------------------------------------------
    # Stratified 5-fold CV
    # --------------------------------------------------------

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=base_seed,
    )

    splits = []

    for k, (
        train_idx,
        test_idx,
    ) in enumerate(
        skf.split(
            X_df,
            y_all,
        )
    ):

        seed = base_seed + k

        # ----------------------------------------------------
        # Raw split
        # ----------------------------------------------------

        X_train_df = (
            X_df
            .iloc[train_idx]
            .copy()
        )

        X_test_df = (
            X_df
            .iloc[test_idx]
            .copy()
        )

        y_train = y_all[train_idx]
        y_test = y_all[test_idx]

        # ----------------------------------------------------
        # Separate variable types
        # ----------------------------------------------------

        X_train_bin_raw = (
            X_train_df[BINARY_COLS]
            .copy()
        )

        X_test_bin_raw = (
            X_test_df[BINARY_COLS]
            .copy()
        )

        X_train_cat = (
            X_train_df[CATEGORICAL_COLS]
            .copy()
        )

        X_test_cat = (
            X_test_df[CATEGORICAL_COLS]
            .copy()
        )

        X_train_cont = (
            X_train_df[CONTINUOUS_COLS]
            .copy()
        )

        X_test_cont = (
            X_test_df[CONTINUOUS_COLS]
            .copy()
        )

        # ====================================================
        # Binary variables
        # ====================================================

        imputer_bin = SimpleImputer(
            strategy="most_frequent"
        )

        X_train_bin = pd.DataFrame(
            imputer_bin.fit_transform(
                X_train_bin_raw
            ),
            columns=BINARY_COLS,
            index=X_train_bin_raw.index,
        ).astype(float)

        X_test_bin = pd.DataFrame(
            imputer_bin.transform(
                X_test_bin_raw
            ),
            columns=BINARY_COLS,
            index=X_test_bin_raw.index,
        ).astype(float)

        # ====================================================
        # Continuous variables
        # ====================================================

        imputer_cont = SimpleImputer(
            strategy="median"
        )

        scaler_cont = StandardScaler()

        # Fit imputation ONLY on training data
        X_train_cont_i = pd.DataFrame(
            imputer_cont.fit_transform(
                X_train_cont
            ),
            columns=CONTINUOUS_COLS,
            index=X_train_cont.index,
        )

        X_test_cont_i = pd.DataFrame(
            imputer_cont.transform(
                X_test_cont
            ),
            columns=CONTINUOUS_COLS,
            index=X_test_cont.index,
        )

        # ====================================================
        # Optional engineered continuous features
        # ====================================================

        (
            X_train_eng,
            engineered_cont_cols,
        ) = _add_engineered_features_after_imputation(
            X_train_cont_i,
            X_train_bin,
            fe_level,
        )

        (
            X_test_eng,
            _,
        ) = _add_engineered_features_after_imputation(
            X_test_cont_i,
            X_test_bin,
            fe_level,
        )

        X_train_cont_full = pd.concat(
            [
                X_train_cont_i,
                X_train_eng,
            ],
            axis=1,
        )

        X_test_cont_full = pd.concat(
            [
                X_test_cont_i,
                X_test_eng,
            ],
            axis=1,
        )

        cont_full_cols = (
            X_train_cont_full
            .columns
            .tolist()
        )

        # Fit scaling ONLY on training data
        X_train_cont_s = pd.DataFrame(
            scaler_cont.fit_transform(
                X_train_cont_full
            ),
            columns=cont_full_cols,
            index=X_train_cont_full.index,
        )

        X_test_cont_s = pd.DataFrame(
            scaler_cont.transform(
                X_test_cont_full
            ),
            columns=cont_full_cols,
            index=X_test_cont_full.index,
        )

        # ====================================================
        # Education
        # ====================================================

        if fe_level == "mild":

            (
                X_train_cat_enc,
                X_test_cat_enc,
                cat_encoded_cols,
                imputer_edu,
            ) = _education_no_ohe_train_test(
                X_train_cat,
                X_test_cat,
            )

            ohe_cols = []

        else:

            (
                X_train_cat_enc,
                X_test_cat_enc,
                ohe_cols,
            ) = _ohe_education_train_test(
                X_train_cat,
                X_test_cat,
            )

            cat_encoded_cols = ohe_cols
            imputer_edu = None

        # ====================================================
        # Combine final predictors
        # ====================================================

        X_train_final_df = pd.concat(
            [
                X_train_bin,
                X_train_cat_enc,
                X_train_cont_s,
            ],
            axis=1,
        )

        X_test_final_df = pd.concat(
            [
                X_test_bin,
                X_test_cat_enc,
                X_test_cont_s,
            ],
            axis=1,
        )

        feature_cols = (
            X_train_final_df
            .columns
            .tolist()
        )

        # Ensure exact train/test correspondence
        assert (
            X_train_final_df.columns.tolist()
            == X_test_final_df.columns.tolist()
        )

        # Ensure preprocessing removed all missing values
        assert not X_train_final_df.isna().any().any()
        assert not X_test_final_df.isna().any().any()

        # ====================================================
        # NumPy arrays
        # ====================================================

        X_train = (
            X_train_final_df
            .to_numpy(dtype=np.float32)
        )

        X_test = (
            X_test_final_df
            .to_numpy(dtype=np.float32)
        )

        # ====================================================
        # Torch tensors
        # ====================================================

        X_train_t = torch.tensor(
            X_train,
            dtype=torch.float32,
            device=device,
        )

        X_test_t = torch.tensor(
            X_test,
            dtype=torch.float32,
            device=device,
        )

        y_train_t = torch.tensor(
            y_train.reshape(-1, 1),
            dtype=torch.float32,
            device=device,
        )

        y_test_t = torch.tensor(
            y_test.reshape(-1, 1),
            dtype=torch.float32,
            device=device,
        )

        # ====================================================
        # Store split
        # ====================================================

        splits.append(
            SplitPack(
                run_id=k,
                seed=seed,
                feature_cols=feature_cols,

                X_train=X_train,
                X_test=X_test,

                y_train=y_train,
                y_test=y_test,

                X_train_t=X_train_t,
                X_test_t=X_test_t,

                y_train_t=y_train_t,
                y_test_t=y_test_t,

                imputer_bin=imputer_bin,
                imputer_cont=imputer_cont,
                scaler_cont=scaler_cont,

                binary_cols=BINARY_COLS.copy(),
                categorical_cols=CATEGORICAL_COLS.copy(),
                continuous_cols=CONTINUOUS_COLS.copy(),

                engineered_cont_cols=(
                    engineered_cont_cols
                ),

                ohe_cols=ohe_cols,

                y_train_mean=float(
                    np.mean(y_train)
                ),

                y_test_mean=float(
                    np.mean(y_test)
                ),

                fe_level=fe_level,
            )
        )

    return splits

# Step 3 — XGBoost globals and setup

This step defines the XGBoost-specific settings used for the Framingham experiments.

For Framingham, the evaluated decision bands are defined dynamically per split using the training-set prevalence:

1. 0.5 × prevalence
2. prevalence
3. 2 × prevalence

In [6]:
import random

import numpy as np
import torch
import xgboost as xgb

from nbloss.trainer import set_seed

from nbloss.xgboost_objectives import (
    make_xgb_smooth_net_benefit_objective,
    make_xgb_smooth_net_benefit_range_objective,
    net_benefit_hard_band_xgb,
)


DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)

SEED_GLOBAL = 1234
MODEL_SEED = 4242

set_seed(SEED_GLOBAL)
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)

BAND_HALF_WIDTH = 0.025


def band_from_t_ref(
    t_ref: float,
    half_width: float = BAND_HALF_WIDTH,
) -> tuple[float, float]:
    eps = 1e-9

    t_ref = float(
        np.clip(
            float(t_ref),
            eps,
            1.0 - eps,
        )
    )

    t_min = max(
        eps,
        t_ref - float(half_width),
    )

    t_max = min(
        1.0 - eps,
        t_ref + float(half_width),
    )

    if not t_min < t_max:
        span = min(
            t_ref - eps,
            1.0 - eps - t_ref,
            float(half_width),
        )

        t_min = t_ref - span
        t_max = t_ref + span

    return float(t_min), float(t_max)


def make_framingham_band_specs(
    prevalence: float,
) -> list[tuple[str, float, float, float]]:
    prevalence = float(prevalence)

    t_ref_half = 0.5 * prevalence
    t_ref_prev = prevalence
    t_ref_double = min(2.0 * prevalence, 0.999)

    return [
        (
            "0.5_x_prev",
            float(t_ref_half),
            *band_from_t_ref(t_ref_half),
        ),
        (
            "prev",
            float(t_ref_prev),
            *band_from_t_ref(t_ref_prev),
        ),
        (
            "2_x_prev",
            float(t_ref_double),
            *band_from_t_ref(t_ref_double),
        ),
    ]


# ---- XGB BCE baseline control
BCE_NUM_BOOST_ROUND = 1000
BCE_EARLY_STOPPING_ROUNDS = 50


# ---- Local calibration settings for BCE-XGB
LOCAL_START_HALF_WIDTH = 0.1
LOCAL_EXPAND_STEP = 0.01
LOCAL_MIN_POS = 50
LOCAL_MIN_NEG = 50

TEMP_MAX_ITER = 500
PLATT_MAX_ITER = 500


# ---- XGB NB stage control
NB_ADDITIONAL_TREES_CAP = 600
NB_EARLY_STOP_TREES = 50
NB_EVAL_EVERY_TREES = 10


# ---- NB grids
TRAIN_RANGE_POINTS = 3
VAL_RANGE_POINTS = 5
TEST_RANGE_POINTS = 201


# ---- Annealed NB inverse temperatures
# Equivalent to old TEMPS = (1.0, 0.25, 0.1)
# because inv_temp = 1 / temp
INV_TEMPS = (1.0, 4.0, 10.0)
NB_ANNEAL_INV_TEMPS = INV_TEMPS


# ---- XGB Hessian variants
HESSIAN_MODES = ["absw", "fixed025", "true"]

# Step 4 — XGB baseline logloss selection and refit

This step defines the baseline XGBoost training procedure.

The baseline model is selected using validation logloss only. After selecting the best hyperparameters and number of trees on an inner validation split, the model is refit on the full available training data using the selected fixed number of trees.

At this stage, no temperature scaling or Platt scaling is applied, so the code matches the earlier uncalibrated baseline method.

In [7]:
# ======================================================================
# Step 5 — XGB baseline logloss selection and refit
# ======================================================================

import xgboost as xgb
import numpy as np



NUM_BOOST_ROUND_CAP = 4000
EARLY_STOP_ROUNDS = 100


def _normalize_xgb_params(params: dict) -> dict:
    p = dict(params)
    if "learning_rate" in p:
        p["eta"] = float(p["learning_rate"])
    if "eta" in p and "learning_rate" not in p:
        p["learning_rate"] = float(p["eta"])
    return p


def baseline_hyperparameter_grid():
    grid = []
    for max_depth in [3, 4]:
        for learning_rate in [0.05, 0.10]:
            for min_child_weight in [1.0, 5.0]:
                for reg_lambda in [1.0, 5.0]:
                    for gamma in [0.0, 1.0]:
                        for subsample in [1.0]:
                            grid.append({
                                "max_depth": int(max_depth),
                                "learning_rate": float(learning_rate),
                                "min_child_weight": float(min_child_weight),
                                "subsample": float(subsample),
                                "colsample_bytree": 1.0,
                                "reg_lambda": float(reg_lambda),
                                "reg_alpha": 0.0,
                                "gamma": float(gamma),
                                "tree_method": "hist",
                            })
    return grid


def xgb_fit_baseline_with_es(
    X_train_enc,
    y_train,
    X_valid_enc,
    y_valid,
    *,
    params: dict,
    seed: int,
    num_boost_round_cap: int = NUM_BOOST_ROUND_CAP,
    early_stopping_rounds: int = EARLY_STOP_ROUNDS,
):
    p = _normalize_xgb_params(params)
    p = dict(p)
    p.setdefault("objective", "binary:logistic")
    p.setdefault("eval_metric", "logloss")
    p.setdefault("tree_method", "hist")
    p.setdefault("seed", int(seed))
    p.setdefault("random_state", int(seed))

    dtrain = xgb.DMatrix(X_train_enc, label=y_train)
    dvalid = xgb.DMatrix(X_valid_enc, label=y_valid)

    booster = xgb.train(
        params=p,
        dtrain=dtrain,
        num_boost_round=int(num_boost_round_cap),
        evals=[(dvalid, "valid")],
        verbose_eval=False,
        early_stopping_rounds=int(early_stopping_rounds),
    )

    best_iter = getattr(booster, "best_iteration", None)
    best_score = getattr(booster, "best_score", None)

    if best_iter is None:
        best_iter = int(num_boost_round_cap) - 1

    if best_score is None:
        preds = np.clip(booster.predict(dvalid), 1e-12, 1.0 - 1e-12)
        best_score = float(
            -np.mean(y_valid * np.log(preds) + (1.0 - y_valid) * np.log(1.0 - preds))
        )

    best_trees = int(best_iter) + 1
    best_logloss = float(best_score)

    return booster, best_trees, best_logloss


def select_baseline_by_val_logloss_with_broadcast(
    X_train_enc,
    y_train,
    X_valid_enc,
    y_valid,
    *,
    seed: int,
):
    grid = baseline_hyperparameter_grid()
    best = None

    print(
        f"\n[Inner selection][Baseline XGB] "
        f"Evaluating {len(grid)} configs (metric=VAL logloss) ..."
    )

    rows = []

    for i, cfg in enumerate(grid, start=1):
        booster, best_trees, val_logloss = xgb_fit_baseline_with_es(
            X_train_enc,
            y_train,
            X_valid_enc,
            y_valid,
            params=cfg,
            seed=seed,
        )

        rows.append({
            "grid_i": int(i),
            "val_logloss": float(val_logloss),
            "best_trees": int(best_trees),
            "params": dict(cfg),
        })

        print(
            f"[Baseline {i:03d}/{len(grid)}] "
            f"val_logloss={val_logloss:.6f} | best_trees={best_trees:4d} | "
            f"max_depth={cfg['max_depth']} | lr={cfg['learning_rate']:.2g} | "
            f"min_child_weight={cfg['min_child_weight']} | "
            f"reg_lambda={cfg['reg_lambda']} | gamma={cfg['gamma']} | "
            f"subsample={cfg['subsample']} | colsample={cfg['colsample_bytree']}"
        )

        if (best is None) or (val_logloss < best[0]):
            best = (float(val_logloss), dict(cfg), int(best_trees))

    best_val_logloss, best_params, best_trees = best

    print(
        f"\n[Baseline BEST] val_logloss={best_val_logloss:.6f} | "
        f"best_trees={best_trees} | params={best_params}"
    )

    return best_params, best_trees, best_val_logloss, rows


def refit_baseline_on_trainval(
    X_trainval_enc,
    y_trainval,
    *,
    params: dict,
    n_trees: int,
    seed: int,
):
    p = _normalize_xgb_params(params)
    p = dict(p)
    p.setdefault("objective", "binary:logistic")
    p.setdefault("eval_metric", "logloss")
    p.setdefault("tree_method", "hist")
    p.setdefault("seed", int(seed))
    p.setdefault("random_state", int(seed))

    dtrainval = xgb.DMatrix(X_trainval_enc, label=y_trainval)

    booster = xgb.train(
        params=p,
        dtrain=dtrainval,
        num_boost_round=int(n_trees),
        verbose_eval=False,
    )

    return booster

# Step 5 — XGB NB selection with annealing

This step continues training from the selected BCE baseline using the Smooth Net Benefit objective.

The Net Benefit stage uses an annealed temperature schedule:

1. temp = 1.0
2. temp = 0.25
3. temp = 0.10

After each temperature stage, the additional trees are committed only if validation hard Net Benefit over the decision range improves. Otherwise, that stage is reverted.

The number of committed trees per temperature stage is stored so that the final model can later be refit cleanly on the full training data.

In [8]:
import numpy as np
import pandas as pd
import xgboost as xgb

from nbloss.xgboost_objectives import (
    make_xgb_smooth_net_benefit_range_objective,
    net_benefit_hard_band_xgb,
)


NB_ANNEAL_INV_TEMPS = (1.0, 4.0, 10.0)

NB_PATIENCE_BY_INV_TEMP = {
    1.0: 40,
    4.0: 20,
    10.0: 20,
}

NB_ADDITIONAL_TREES_CAP = int(NB_ADDITIONAL_TREES_CAP)
NB_EVAL_EVERY_TREES = int(NB_EVAL_EVERY_TREES)


def _inv_temp_suffix(inv_temp: float) -> str:
    if np.isclose(inv_temp, 1.0):
        return "it1"
    if np.isclose(inv_temp, 4.0):
        return "it4"
    if np.isclose(inv_temp, 10.0):
        return "it10"
    return f"it{str(inv_temp).replace('.', '')}"


def nb_hyperparameter_grid_annealed():
    grid = []

    for max_depth in [3, 4]:
        for learning_rate in [0.05, 0.10]:
            for min_child_weight in [1.0, 5.0]:
                for reg_lambda in [1.0, 5.0]:
                    for gamma in [0.0, 1.0]:
                        for subsample in [1.0]:
                            grid.append({
                                "anneal_inv_temps": tuple(NB_ANNEAL_INV_TEMPS),
                                "max_depth": int(max_depth),
                                "learning_rate": float(learning_rate),
                                "min_child_weight": float(min_child_weight),
                                "reg_lambda": float(reg_lambda),
                                "reg_alpha": 0.0,
                                "gamma": float(gamma),
                                "subsample": float(subsample),
                                "colsample_bytree": 1.0,
                                "additional_trees_cap": int(NB_ADDITIONAL_TREES_CAP),
                                "eval_every_trees": int(NB_EVAL_EVERY_TREES),
                                "patience_it1": 40,
                                "patience_it4": 20,
                                "patience_it10": 20,
                            })

    return grid


def nb_hyperparameter_grid_fixed_temp():
    return nb_hyperparameter_grid_annealed()


def warm_start_baseline_train_only(
    X_train_enc,
    y_train,
    *,
    baseline_params: dict,
    baseline_trees: int,
    seed: int = SEED_GLOBAL,
):
    p = _normalize_xgb_params(dict(baseline_params))
    p.setdefault("objective", "binary:logistic")
    p.setdefault("eval_metric", "logloss")
    p.setdefault("tree_method", "hist")
    p.setdefault("seed", int(seed))
    p.setdefault("random_state", int(seed))

    dtrain = xgb.DMatrix(
        X_train_enc,
        label=np.asarray(y_train, dtype=np.float32),
    )

    booster = xgb.train(
        params=p,
        dtrain=dtrain,
        num_boost_round=int(baseline_trees),
        verbose_eval=False,
    )

    return booster


def _make_nb_stage_params_from_baseline(
    baseline_params: dict,
    nb_cfg: dict,
    *,
    seed: int,
):
    p = dict(baseline_params)

    p["max_depth"] = int(nb_cfg["max_depth"])
    p["learning_rate"] = float(nb_cfg["learning_rate"])
    p["min_child_weight"] = float(nb_cfg["min_child_weight"])
    p["reg_lambda"] = float(nb_cfg["reg_lambda"])
    p["reg_alpha"] = float(nb_cfg["reg_alpha"])
    p["gamma"] = float(nb_cfg["gamma"])
    p["subsample"] = float(nb_cfg["subsample"])
    p["colsample_bytree"] = float(nb_cfg["colsample_bytree"])

    p["objective"] = "binary:logistic"
    p["tree_method"] = "hist"
    p["seed"] = int(seed)
    p["random_state"] = int(seed)

    return _normalize_xgb_params(p)


def _patience_for_inv_temp(inv_temp: float, cfg: dict) -> int:
    suffix = _inv_temp_suffix(inv_temp)
    return int(cfg.get(f"patience_{suffix}", 20))


def nb_continue_annealed_with_stage_commit(
    *,
    booster0,
    X_train_enc,
    y_train,
    X_valid_enc,
    y_valid,
    t_min,
    t_max,
    anneal_inv_temps,
    num_points_train,
    n_grid_eval,
    additional_trees_cap,
    eval_every_trees,
    params_nb_stage,
    hessian_mode,
    cfg,
):
    dtrain = xgb.DMatrix(
        X_train_enc,
        label=np.asarray(y_train, dtype=np.float32),
    )

    dvalid = xgb.DMatrix(
        X_valid_enc,
        label=np.asarray(y_valid, dtype=np.float32),
    )

    eval_every_trees = int(max(1, eval_every_trees))
    additional_trees_cap = int(additional_trees_cap)

    margins0 = booster0.predict(
        dvalid,
        output_margin=True,
    )

    global_best_nb = net_benefit_hard_band_xgb(
        margins0,
        y_valid,
        t_min=float(t_min),
        t_max=float(t_max),
        n_grid=int(n_grid_eval),
        tau_eval=1.0,
        input_is_margin=True,
    )

    global_best_booster = booster0
    total_added_trained = 0
    total_committed_added = 0
    stage_info = []

    for inv_temp in anneal_inv_temps:
        inv_temp = float(inv_temp)

        suffix = _inv_temp_suffix(inv_temp)
        patience_checks = _patience_for_inv_temp(inv_temp, cfg)

        objective = make_xgb_smooth_net_benefit_range_objective(
            t_min=float(t_min),
            t_max=float(t_max),
            num_points=int(num_points_train),
            inv_temp=float(inv_temp),
            hessian_mode=str(hessian_mode),
        )

        stage_booster = global_best_booster
        stage_best_booster = global_best_booster
        stage_best_nb = float(global_best_nb)

        stage_added_trained = 0
        stage_best_added = 0
        no_improve = 0

        # IMPORTANT:
        # The cap is now PER inverse-temperature stage,
        # not shared globally across the whole annealing schedule.
        while stage_added_trained < additional_trees_cap:
            step = min(
                eval_every_trees,
                additional_trees_cap - stage_added_trained,
            )

            stage_booster = xgb.train(
                params=params_nb_stage,
                dtrain=dtrain,
                num_boost_round=int(step),
                obj=objective,
                xgb_model=stage_booster,
                verbose_eval=False,
            )

            stage_added_trained += int(step)
            total_added_trained += int(step)

            margins = stage_booster.predict(
                dvalid,
                output_margin=True,
            )

            val_nb = net_benefit_hard_band_xgb(
                margins,
                y_valid,
                t_min=float(t_min),
                t_max=float(t_max),
                n_grid=int(n_grid_eval),
                tau_eval=1.0,
                input_is_margin=True,
            )

            if val_nb > stage_best_nb + 1e-10:
                stage_best_nb = float(val_nb)
                stage_best_booster = stage_booster
                stage_best_added = int(stage_added_trained)
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= patience_checks:
                break

        improved_global = stage_best_nb > global_best_nb + 1e-10

        if improved_global:
            global_best_nb = float(stage_best_nb)
            global_best_booster = stage_best_booster
            committed_added_this_stage = int(stage_best_added)
        else:
            committed_added_this_stage = 0

        total_committed_added += int(committed_added_this_stage)

        stage_info.append({
            "inv_temp": float(inv_temp),
            "suffix": suffix,
            "patience_checks": int(patience_checks),
            "stage_added_trained": int(stage_added_trained),
            "stage_best_added": int(stage_best_added),
            "committed_added": int(committed_added_this_stage),
            "stage_best_val_nb": float(stage_best_nb),
            "global_best_val_nb_after_stage": float(global_best_nb),
            "improved_global": bool(improved_global),
        })

        print(
            f"    [anneal inv_temp={inv_temp:g}] "
            f"stage_best_nb={stage_best_nb:+.6f} | "
            f"stage_best_added={stage_best_added:4d} | "
            f"committed_added={committed_added_this_stage:4d} | "
            f"committed={improved_global} | "
            f"global_best_nb={global_best_nb:+.6f}"
        )

    return (
        global_best_booster,
        int(total_committed_added),
        float(global_best_nb),
        stage_info,
    )

# Step 6 — Inner CV selection: baseline logloss → annealed NB

This step performs inner cross-validation on the outer-training data.

First, the BCE/logloss XGBoost baseline is selected by validation logloss. Then, starting from that selected baseline, the annealed Smooth Net Benefit stage is selected by validation hard Net Benefit over the threshold band.

For final refitting later, the selected number of baseline trees and NB-added trees are summarized using the median across inner folds.

In [9]:

from sklearn.model_selection import StratifiedKFold

import numpy as np
import pandas as pd
import xgboost as xgb

from nbloss.xgboost_objectives import (
    make_xgb_smooth_net_benefit_range_objective,
    net_benefit_hard_band_xgb,
)


def make_inner_cv_folds(
    y_train: np.ndarray,
    *,
    seed: int,
    n_splits: int = 5,
):
    y_train = np.asarray(y_train).astype(int)

    skf = StratifiedKFold(
        n_splits=int(n_splits),
        shuffle=True,
        random_state=int(seed),
    )

    return [
        (tr, va)
        for tr, va in skf.split(np.zeros_like(y_train), y_train)
    ]


def select_baseline_and_nb_inner_cv(
    X_train_enc: np.ndarray,
    y_train: np.ndarray,
    *,
    t_min: float,
    t_max: float,
    hessian_mode: str,
    inner_cv_splits: int = 5,
    seed: int = SEED_GLOBAL,
):
    """
    Select baseline XGBoost and annealed NB-XGBoost by inner CV.

    Returns
    -------
    best_baseline_params:
        Hyperparameters selected by validation logloss.

    median_baseline_trees:
        Median number of BCE baseline trees across inner folds.

    best_nb_cfg:
        NB-stage hyperparameters selected by validation hard NB.
        Also includes:
        - _baseline_params_for_nb_stage
        - median_added_trees
        - median_added_trees_it1
        - median_added_trees_it4
        - median_added_trees_it10

    median_added_trees:
        Median total number of committed NB trees across inner folds.
    """
    X_train_enc = np.asarray(X_train_enc, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.float32)

    folds = make_inner_cv_folds(
        y_train,
        seed=int(seed) + 111,
        n_splits=int(inner_cv_splits),
    )

    # =========================================================
    # 1) Baseline selection by validation logloss
    # =========================================================

    base_grid = baseline_hyperparameter_grid()
    best_base = None
    best_base_fold_trees = None

    print(
        f"\n[Inner CV][Baseline] {len(base_grid)} configs | "
        f"metric=VAL logloss | folds={inner_cv_splits}"
    )

    for gi, params in enumerate(base_grid, start=1):
        fold_logloss = []
        fold_trees = []
        fold_nb_info = []

        for tr_i, va_i in folds:
            X_tr = X_train_enc[tr_i]
            y_tr = y_train[tr_i]

            X_va = X_train_enc[va_i]
            y_va = y_train[va_i]

            booster, best_trees, val_logloss = xgb_fit_baseline_with_es(
                X_tr,
                y_tr,
                X_va,
                y_va,
                params=params,
                seed=int(seed),
            )

            fold_logloss.append(float(val_logloss))
            fold_trees.append(int(best_trees))

            dvalid = xgb.DMatrix(X_va, label=y_va)
            margins = booster.predict(dvalid, output_margin=True)

            nb_val = net_benefit_hard_band_xgb(
                margins,
                y_va,
                t_min=float(t_min),
                t_max=float(t_max),
                n_grid=int(VAL_RANGE_POINTS),
                tau_eval=1.0,
                input_is_margin=True,
            )

            fold_nb_info.append(float(nb_val))

        mean_logloss = float(np.mean(fold_logloss))
        median_trees = int(np.median(fold_trees))
        mean_nb_info = float(np.mean(fold_nb_info))

        print(
            f"[Inner CV][Baseline {gi:03d}/{len(base_grid)}] "
            f"val_logloss={mean_logloss:.6f} | "
            f"val_nb(info)={mean_nb_info:+.6f} | "
            f"median_trees={median_trees:4d} | "
            f"max_depth={params['max_depth']} | "
            f"lr={params['learning_rate']:.2g} | "
            f"min_child_weight={params['min_child_weight']} | "
            f"reg_lambda={params['reg_lambda']} | "
            f"gamma={params['gamma']} | "
            f"subsample={params['subsample']} | "
            f"colsample={params['colsample_bytree']}"
        )

        if (best_base is None) or (mean_logloss < best_base[0]):
            best_base = (
                float(mean_logloss),
                dict(params),
                int(median_trees),
                float(mean_nb_info),
            )
            best_base_fold_trees = list(fold_trees)

    best_baseline_params = best_base[1]
    median_baseline_trees = int(np.median(best_base_fold_trees))

    print(
        f"\n[Inner CV][Baseline BEST] "
        f"val_logloss={best_base[0]:.6f} | "
        f"val_nb(info)={best_base[3]:+.6f} | "
        f"median_trees={median_baseline_trees} | "
        f"params={best_baseline_params}"
    )

    print(
        f"[Inner CV][Baseline BEST] "
        f"fold_specific_best_trees={best_base_fold_trees}"
    )

    # =========================================================
    # 2) Annealed NB selection by validation hard NB
    # =========================================================

    nb_grid = nb_hyperparameter_grid_annealed()
    best_nb = None

    print(
        f"\n[Inner CV][Annealed NB | h={hessian_mode}] "
        f"{len(nb_grid)} configs | metric=VAL hard NB | folds={inner_cv_splits}"
    )

    for gi, cfg in enumerate(nb_grid, start=1):
        fold_val_nb = []
        fold_added_total = []
        fold_added_by_inv_temp = {
            1.0: [],
            4.0: [],
            10.0: [],
        }

        for fj, (tr_i, va_i) in enumerate(folds):
            X_tr = X_train_enc[tr_i]
            y_tr = y_train[tr_i]

            X_va = X_train_enc[va_i]
            y_va = y_train[va_i]

            fold_baseline_trees = int(best_base_fold_trees[fj])

            warm0 = warm_start_baseline_train_only(
                X_tr,
                y_tr,
                baseline_params=best_baseline_params,
                baseline_trees=fold_baseline_trees,
                seed=int(seed),
            )

            params_nb_stage = _make_nb_stage_params_from_baseline(
                best_baseline_params,
                cfg,
                seed=int(seed),
            )

            _, added_best, val_nb, stage_info = nb_continue_annealed_with_stage_commit(
                booster0=warm0,
                X_train_enc=X_tr,
                y_train=y_tr,
                X_valid_enc=X_va,
                y_valid=y_va,
                t_min=float(t_min),
                t_max=float(t_max),
                anneal_inv_temps=tuple(cfg["anneal_inv_temps"]),
                num_points_train=int(TRAIN_RANGE_POINTS),
                n_grid_eval=int(VAL_RANGE_POINTS),
                additional_trees_cap=int(cfg["additional_trees_cap"]),
                eval_every_trees=int(cfg["eval_every_trees"]),
                params_nb_stage=params_nb_stage,
                hessian_mode=str(hessian_mode),
                cfg=cfg,
            )

            fold_val_nb.append(float(val_nb))
            fold_added_total.append(int(added_best))

            committed_by_temp = {
                float(info["inv_temp"]): int(info["committed_added"])
                for info in stage_info
            }

            for inv_temp in fold_added_by_inv_temp:
                fold_added_by_inv_temp[inv_temp].append(
                    int(committed_by_temp.get(float(inv_temp), 0))
                )

        mean_val_nb = float(np.mean(fold_val_nb))
        median_added_total = int(np.median(fold_added_total))

        median_added_it1 = int(np.median(fold_added_by_inv_temp[1.0]))
        median_added_it4 = int(np.median(fold_added_by_inv_temp[4.0]))
        median_added_it10 = int(np.median(fold_added_by_inv_temp[10.0]))

        print(
            f"[Inner CV][Annealed NB {gi:03d}/{len(nb_grid)} | h={hessian_mode}] "
            f"val_nb={mean_val_nb:+.6f} | "
            f"median_added_total={median_added_total:4d} | "
            f"it1={median_added_it1:4d} | "
            f"it4={median_added_it4:4d} | "
            f"it10={median_added_it10:4d} | "
            f"schedule={cfg['anneal_inv_temps']} | "
            f"max_depth={cfg['max_depth']} | "
            f"lr={cfg['learning_rate']:.2g} | "
            f"min_child_weight={cfg['min_child_weight']} | "
            f"reg_lambda={cfg['reg_lambda']} | "
            f"gamma={cfg['gamma']} | "
            f"subsample={cfg['subsample']} | "
            f"colsample={cfg['colsample_bytree']}"
        )

        if (best_nb is None) or (mean_val_nb > best_nb[0]):
            cfg_best = dict(cfg)
            cfg_best["_baseline_params_for_nb_stage"] = dict(best_baseline_params)
            cfg_best["median_added_trees"] = int(median_added_total)
            cfg_best["median_added_trees_it1"] = int(median_added_it1)
            cfg_best["median_added_trees_it4"] = int(median_added_it4)
            cfg_best["median_added_trees_it10"] = int(median_added_it10)

            best_nb = (
                float(mean_val_nb),
                cfg_best,
                int(median_added_total),
            )

    best_nb_cfg = best_nb[1]
    median_added_trees = int(best_nb[2])

    print(
        f"\n[Inner CV][Annealed NB BEST | h={hessian_mode}] "
        f"val_nb={best_nb[0]:+.6f} | "
        f"median_added={median_added_trees} | "
        f"cfg={best_nb_cfg}"
    )

    return (
        best_baseline_params,
        int(median_baseline_trees),
        best_nb_cfg,
        int(median_added_trees),
    )


def refit_nb_on_train_from_baseline_encoded(
    *,
    booster_baseline_train,
    X_train_enc,
    y_train,
    t_min: float,
    t_max: float,
    nb_cfg: dict,
    hessian_mode: str,
    seed: int,
):
    dtrain = xgb.DMatrix(
        X_train_enc,
        label=np.asarray(y_train, dtype=np.float32),
    )

    params_nb = _make_nb_stage_params_from_baseline(
        baseline_params=nb_cfg.get("_baseline_params_for_nb_stage"),
        nb_cfg=nb_cfg,
        seed=int(seed),
    )

    stage_tree_counts = {
        1.0: int(nb_cfg.get("median_added_trees_it1", 0)),
        4.0: int(nb_cfg.get("median_added_trees_it4", 0)),
        10.0: int(nb_cfg.get("median_added_trees_it10", 0)),
    }

    booster = booster_baseline_train

    for inv_temp in nb_cfg.get("anneal_inv_temps", NB_ANNEAL_INV_TEMPS):
        inv_temp = float(inv_temp)
        n_stage_trees = int(stage_tree_counts.get(inv_temp, 0))

        if n_stage_trees <= 0:
            continue

        objective = make_xgb_smooth_net_benefit_range_objective(
            t_min=float(t_min),
            t_max=float(t_max),
            num_points=int(TRAIN_RANGE_POINTS),
            inv_temp=float(inv_temp),
            hessian_mode=str(hessian_mode),
        )

        print(
            f"    [final refit] inv_temp={inv_temp:g} | "
            f"adding {n_stage_trees} trees"
        )

        booster = xgb.train(
            params=params_nb,
            dtrain=dtrain,
            num_boost_round=int(n_stage_trees),
            obj=objective,
            xgb_model=booster,
            verbose_eval=False,
        )

    return booster




## Step 7 — Prediction, local calibration, and Net Benefit evaluation

Defines helper functions for predicting logits, converting logits to probabilities, selecting local calibration subsets, fitting local temperature scaling, fitting local Platt scaling, applying local calibration, and evaluating average Net Benefit over a threshold band.

In [10]:

import numpy as np
import torch
import torch.nn as nn

from nbloss.xgboost_objectives import net_benefit_hard_band_xgb


def sigmoid_np(logits_np: np.ndarray) -> np.ndarray:
    logits_t = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1)
    )

    return (
        torch.sigmoid(logits_t)
        .detach()
        .cpu()
        .numpy()
        .astype(np.float64)
    )


def subset_counts(mask, y_np):
    y_sub = np.asarray(y_np).reshape(-1)[np.asarray(mask, dtype=bool)]

    n_pos = int(np.sum(y_sub == 1))
    n_neg = int(np.sum(y_sub == 0))

    return int(len(y_sub)), n_pos, n_neg


def find_local_calibration_range(
    probs_train: np.ndarray,
    y_train: np.ndarray,
    *,
    t_ref: float,
    start_half_width: float = LOCAL_START_HALF_WIDTH,
    expand_step: float = LOCAL_EXPAND_STEP,
    min_pos: int = LOCAL_MIN_POS,
    min_neg: int = LOCAL_MIN_NEG,
) -> dict:
    probs_train = np.asarray(probs_train, dtype=float).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)

    half_width = float(start_half_width)
    n_expand_steps = 0

    while True:
        low = max(0.0, float(t_ref) - half_width)
        high = min(1.0, float(t_ref) + half_width)

        mask = (probs_train >= low) & (probs_train <= high)

        n, n_pos, n_neg = subset_counts(mask, y_train)

        met_minimum = (
            n_pos >= int(min_pos)
            and n_neg >= int(min_neg)
        )

        used_full_range = (
            low <= 0.0
            and high >= 1.0
        )

        if met_minimum or used_full_range:
            return {
                "low": float(low),
                "high": float(high),
                "half_width": float(half_width),
                "range_width": float(high - low),
                "n_expand_steps": int(n_expand_steps),
                "n": int(n),
                "n_pos": int(n_pos),
                "n_neg": int(n_neg),
                "met_minimum": bool(met_minimum),
                "used_full_range": bool(used_full_range),
                "mask": mask,
            }

        half_width += float(expand_step)
        n_expand_steps += 1


def fit_temperature_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = TEMP_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(
            bce(logits, y)
            .detach()
            .cpu()
            .item()
        )

    log_temperature = torch.zeros(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)

        temperature = torch.exp(log_temperature).clamp(
            min=1e-6,
            max=1e6,
        )

        loss = bce(logits / temperature, y)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite temperature loss: {loss.detach().item()}"
            )

        loss.backward()

        return loss

    optimizer.step(closure)

    with torch.no_grad():
        temperature = torch.exp(log_temperature).clamp(
            min=1e-6,
            max=1e6,
        )

        nll_after = float(
            bce(logits / temperature, y)
            .detach()
            .cpu()
            .item()
        )

    return {
        "temperature": float(temperature.detach().cpu().item()),
        "nll_before": float(nll_before),
        "nll_after": float(nll_after),
    }


def apply_local_temperature_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    temperature: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = (
        np.asarray(logits_np, dtype=np.float64)
        .reshape(-1)
        .copy()
    )

    probs_reference_np = np.asarray(
        probs_reference_np,
        dtype=float,
    ).reshape(-1)

    mask = (
        (probs_reference_np >= float(low))
        & (probs_reference_np <= float(high))
    )

    logits_out[mask] = logits_out[mask] / float(temperature)

    return logits_out, mask


def fit_platt_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = PLATT_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(
            bce(logits, y)
            .detach()
            .cpu()
            .item()
        )

    slope = torch.ones(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    intercept = torch.zeros(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [slope, intercept],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)

        calibrated_logits = slope * logits + intercept
        loss = bce(calibrated_logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite Platt loss: {loss.detach().item()}"
            )

        loss.backward()

        return loss

    optimizer.step(closure)

    with torch.no_grad():
        calibrated_logits = slope * logits + intercept

        nll_after = float(
            bce(calibrated_logits, y)
            .detach()
            .cpu()
            .item()
        )

    return {
        "platt_slope": float(slope.detach().cpu().item()),
        "platt_intercept": float(intercept.detach().cpu().item()),
        "nll_before": float(nll_before),
        "nll_after": float(nll_after),
    }


def apply_local_platt_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    slope: float,
    intercept: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = (
        np.asarray(logits_np, dtype=np.float64)
        .reshape(-1)
        .copy()
    )

    probs_reference_np = np.asarray(
        probs_reference_np,
        dtype=float,
    ).reshape(-1)

    mask = (
        (probs_reference_np >= float(low))
        & (probs_reference_np <= float(high))
    )

    logits_out[mask] = (
        float(slope) * logits_out[mask]
        + float(intercept)
    )

    return logits_out, mask


def evaluate_nb_from_xgb_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    thresh_min: float,
    thresh_max: float,
    num_points: int = TEST_RANGE_POINTS,
) -> float:
    return net_benefit_hard_band_xgb(
        logits_np,
        y_np,
        t_min=float(thresh_min),
        t_max=float(thresh_max),
        n_grid=int(num_points),
        tau_eval=1.0,
        input_is_margin=True,
    )


def evaluate_bce_xgb_local_calibration(
    *,
    train_logits_base: np.ndarray,
    y_train: np.ndarray,
    test_logits_base: np.ndarray,
    y_test: np.ndarray,
    t_ref: float,
    t_min: float,
    t_max: float,
) -> dict:
    """
    Fit local temperature scaling and local Platt scaling on BCE-XGB train logits,
    then evaluate calibrated BCE-XGB logits on the test set.

    The local calibration range is selected using BCE-XGB train probabilities.
    The same probability range is applied to BCE-XGB test probabilities.

    No NB-XGB logits are used or modified here.
    """
    train_logits_base = np.asarray(
        train_logits_base,
        dtype=np.float64,
    ).reshape(-1)

    test_logits_base = np.asarray(
        test_logits_base,
        dtype=np.float64,
    ).reshape(-1)

    y_train = np.asarray(y_train).reshape(-1)
    y_test = np.asarray(y_test).reshape(-1)

    train_probs_base = sigmoid_np(train_logits_base)
    test_probs_base = sigmoid_np(test_logits_base)

    local_range = find_local_calibration_range(
        train_probs_base,
        y_train,
        t_ref=float(t_ref),
    )

    local_mask_train = np.asarray(
        local_range["mask"],
        dtype=bool,
    )

    if local_mask_train.sum() == 0:
        raise RuntimeError(
            "Local calibration mask is empty on the BCE-XGB training set."
        )

    # --------------------------------------------------
    # Uncalibrated BCE-XGB NB
    # --------------------------------------------------
    nb_uncalibrated = evaluate_nb_from_xgb_logits_np(
        test_logits_base,
        y_test,
        thresh_min=float(t_min),
        thresh_max=float(t_max),
        num_points=int(TEST_RANGE_POINTS),
    )

    # --------------------------------------------------
    # Local temperature scaling
    # --------------------------------------------------
    temp_fit = fit_temperature_from_logits_np(
        train_logits_base[local_mask_train],
        y_train[local_mask_train],
    )

    test_logits_temp, test_mask_temp = apply_local_temperature_to_logits(
        test_logits_base,
        test_probs_base,
        low=float(local_range["low"]),
        high=float(local_range["high"]),
        temperature=float(temp_fit["temperature"]),
    )

    nb_temp_scaled = evaluate_nb_from_xgb_logits_np(
        test_logits_temp,
        y_test,
        thresh_min=float(t_min),
        thresh_max=float(t_max),
        num_points=int(TEST_RANGE_POINTS),
    )

    # --------------------------------------------------
    # Local Platt scaling
    # --------------------------------------------------
    platt_fit = fit_platt_from_logits_np(
        train_logits_base[local_mask_train],
        y_train[local_mask_train],
    )

    test_logits_platt, test_mask_platt = apply_local_platt_to_logits(
        test_logits_base,
        test_probs_base,
        low=float(local_range["low"]),
        high=float(local_range["high"]),
        slope=float(platt_fit["platt_slope"]),
        intercept=float(platt_fit["platt_intercept"]),
    )

    nb_platt_scaled = evaluate_nb_from_xgb_logits_np(
        test_logits_platt,
        y_test,
        thresh_min=float(t_min),
        thresh_max=float(t_max),
        num_points=int(TEST_RANGE_POINTS),
    )

    return {
        "local_calib_low": float(local_range["low"]),
        "local_calib_high": float(local_range["high"]),
        "local_calib_half_width": float(local_range["half_width"]),
        "local_calib_n": int(local_range["n"]),
        "local_calib_n_pos": int(local_range["n_pos"]),
        "local_calib_n_neg": int(local_range["n_neg"]),
        "local_calib_met_minimum": bool(local_range["met_minimum"]),
        "local_calib_used_full_range": bool(local_range["used_full_range"]),

        "bce_nb_uncalibrated": float(nb_uncalibrated),

        "temp_temperature": float(temp_fit["temperature"]),
        "temp_nll_before": float(temp_fit["nll_before"]),
        "temp_nll_after": float(temp_fit["nll_after"]),
        "temp_test_mask_n": int(np.sum(test_mask_temp)),
        "bce_nb_temp_scaled": float(nb_temp_scaled),

        "platt_slope": float(platt_fit["platt_slope"]),
        "platt_intercept": float(platt_fit["platt_intercept"]),
        "platt_nll_before": float(platt_fit["nll_before"]),
        "platt_nll_after": float(platt_fit["nll_after"]),
        "platt_test_mask_n": int(np.sum(test_mask_platt)),
        "bce_nb_platt_scaled": float(nb_platt_scaled),
    }

# Step 8 — Create the preprocessing variants

Create the five repeated train/test splits for the two preprocessing variants used in the Framingham experiments. The `mild` variant retains education as a single ordinal feature, whereas the `moderate` variant represents education using one-hot encoding.

In [11]:
splits_mild = make_framingham_splits_5x(
    model_df,
    target_col="FifteenYearCHD",
    base_seed=42,
    n_runs=5,
    device=DEVICE,
    fe_level="mild",
)

splits_moderate = make_framingham_splits_5x(
    model_df,
    target_col="FifteenYearCHD",
    base_seed=42,
    n_runs=5,
    device=DEVICE,
    fe_level="moderate",
)

print(f"Mild preprocessing     : {len(splits_mild)} splits")
print(f"Moderate preprocessing : {len(splits_moderate)} splits")
print(f"Device                 : {DEVICE}")

Mild preprocessing     : 5 splits
Moderate preprocessing : 5 splits
Device                 : cpu


# Step 8 — Display XGBoost results

Running the models including platt and temperature calibration 

In [12]:

import numpy as np
import pandas as pd
import xgboost as xgb


xgb_results = []


DATASETS = {
    "mild": splits_mild,
    "moderate": splits_moderate,
}


def add_xgb_result_row(
    *,
    dataset_name: str,
    run_id: int,
    band_name: str,
    model_name: str,
    split_pack,
    t_ref: float,
    t_min: float,
    t_max: float,
    test_nb: float,
    delta_vs_bce: float,
    hessian_mode: str = "__BASELINE__",
    baseline_trees: int | None = None,
    baseline_params: dict | None = None,
    nb_added_trees: int | None = None,
    nb_added_trees_it1: int | None = None,
    nb_added_trees_it4: int | None = None,
    nb_added_trees_it10: int | None = None,
    nb_anneal_schedule=None,
    nb_params: dict | None = None,
    calibration_info: dict | None = None,
):
    calibration_info = calibration_info or {}

    xgb_results.append({
        "dataset": str(dataset_name),
        "fe_level": str(dataset_name),
        "run_id": int(run_id),
        "band_name": str(band_name),
        "run_type": str(band_name),
        "model": str(model_name),
        "hessian_mode": str(hessian_mode),

        "prev_train": float(split_pack.y_train_mean),
        "prev_test": float(split_pack.y_test_mean),

        "t_ref": float(t_ref),
        "threshold": float(t_ref),
        "t_min": float(t_min),
        "t_max": float(t_max),
        "band_min": float(t_min),
        "band_max": float(t_max),

        "test_nb": float(test_nb),
        "baseline_test_nb": (
            float(test_nb)
            if model_name == "bce_xgb"
            else np.nan
        ),
        "nb_test_nb": (
            float(test_nb)
            if model_name == "nb_xgb"
            else np.nan
        ),
        "delta_vs_bce": float(delta_vs_bce),
        "delta_nb_vs_baseline": (
            float(delta_vs_bce)
            if model_name == "nb_xgb"
            else np.nan
        ),

        "baseline_trees": (
            int(baseline_trees)
            if baseline_trees is not None
            else np.nan
        ),
        "baseline_params": (
            str(baseline_params)
            if baseline_params is not None
            else None
        ),

        "nb_added_trees": (
            int(nb_added_trees)
            if nb_added_trees is not None
            else np.nan
        ),
        "nb_added_trees_it1": (
            int(nb_added_trees_it1)
            if nb_added_trees_it1 is not None
            else np.nan
        ),
        "nb_added_trees_it4": (
            int(nb_added_trees_it4)
            if nb_added_trees_it4 is not None
            else np.nan
        ),
        "nb_added_trees_it10": (
            int(nb_added_trees_it10)
            if nb_added_trees_it10 is not None
            else np.nan
        ),
        "nb_anneal_schedule": (
            str(nb_anneal_schedule)
            if nb_anneal_schedule is not None
            else None
        ),
        "nb_params": (
            str(nb_params)
            if nb_params is not None
            else None
        ),

        "local_calib_low": calibration_info.get("local_calib_low", np.nan),
        "local_calib_high": calibration_info.get("local_calib_high", np.nan),
        "local_calib_half_width": calibration_info.get("local_calib_half_width", np.nan),
        "local_calib_n": calibration_info.get("local_calib_n", np.nan),
        "local_calib_n_pos": calibration_info.get("local_calib_n_pos", np.nan),
        "local_calib_n_neg": calibration_info.get("local_calib_n_neg", np.nan),
        "local_calib_met_minimum": calibration_info.get("local_calib_met_minimum", np.nan),
        "local_calib_used_full_range": calibration_info.get("local_calib_used_full_range", np.nan),

        # Keep these names: this is BCE post-hoc temperature scaling,
        # not smooth-NB inverse-temperature training.
        "temperature": calibration_info.get("temp_temperature", np.nan),
        "temp_nll_before": calibration_info.get("temp_nll_before", np.nan),
        "temp_nll_after": calibration_info.get("temp_nll_after", np.nan),
        "temp_test_mask_n": calibration_info.get("temp_test_mask_n", np.nan),

        "platt_slope": calibration_info.get("platt_slope", np.nan),
        "platt_intercept": calibration_info.get("platt_intercept", np.nan),
        "platt_nll_before": calibration_info.get("platt_nll_before", np.nan),
        "platt_nll_after": calibration_info.get("platt_nll_after", np.nan),
        "platt_test_mask_n": calibration_info.get("platt_test_mask_n", np.nan),

        "n_features": int(len(split_pack.feature_cols)),
    })


def run_one_outer_rotation_xgb_with_bce_calibration(
    *,
    X_train_enc: np.ndarray,
    y_train: np.ndarray,
    X_test_enc: np.ndarray,
    y_test: np.ndarray,
    run_id: int,
    band_name: str,
    t_ref: float,
    t_min: float,
    t_max: float,
    fe_level: str,
    split_pack,
    hessian_modes: list[str] | None = None,
    inner_cv_splits: int = 5,
    seed: int = SEED_GLOBAL,
):
    if hessian_modes is None:
        hessian_modes = list(HESSIAN_MODES)

    X_train_enc = np.asarray(X_train_enc, dtype=np.float32)
    X_test_enc = np.asarray(X_test_enc, dtype=np.float32)

    y_train = np.asarray(y_train, dtype=np.float32).reshape(-1)
    y_test = np.asarray(y_test, dtype=np.float32).reshape(-1)

    selected_by_hessian = {}

    # --------------------------------------------------
    # Select baseline + NB configuration for each Hessian mode
    # --------------------------------------------------
    for h in hessian_modes:
        (
            best_baseline_params,
            median_baseline_trees,
            best_nb_cfg,
            median_added_trees,
        ) = select_baseline_and_nb_inner_cv(
            X_train_enc,
            y_train,
            t_min=float(t_min),
            t_max=float(t_max),
            hessian_mode=str(h),
            inner_cv_splits=int(inner_cv_splits),
            seed=int(seed),
        )

        selected_by_hessian[str(h)] = {
            "best_baseline_params": dict(best_baseline_params),
            "median_baseline_trees": int(median_baseline_trees),
            "best_nb_cfg": dict(best_nb_cfg),
            "median_added_trees": int(median_added_trees),
        }

    # --------------------------------------------------
    # Refit one BCE baseline on full outer-training data
    # Use first Hessian mode only to define shared baseline
    # --------------------------------------------------
    first_h = str(hessian_modes[0])

    best_baseline_params = selected_by_hessian[first_h]["best_baseline_params"]
    median_baseline_trees = selected_by_hessian[first_h]["median_baseline_trees"]

    booster_baseline_train = refit_baseline_on_trainval(
        X_train_enc,
        y_train,
        params=best_baseline_params,
        n_trees=int(median_baseline_trees),
        seed=int(seed),
    )

    dtrain = xgb.DMatrix(X_train_enc, label=y_train)
    dtest = xgb.DMatrix(X_test_enc, label=y_test)

    train_logits_base = booster_baseline_train.predict(
        dtrain,
        output_margin=True,
    )

    test_logits_base = booster_baseline_train.predict(
        dtest,
        output_margin=True,
    )

    # --------------------------------------------------
    # BCE-XGB uncalibrated test NB
    # --------------------------------------------------
    nb_bce = evaluate_nb_from_xgb_logits_np(
        test_logits_base,
        y_test,
        thresh_min=float(t_min),
        thresh_max=float(t_max),
        num_points=int(TEST_RANGE_POINTS),
    )

    print(
        f"\n[TEST][{fe_level} | run {run_id} | {band_name}] "
        f"BCE-XGB NB={nb_bce:+.6f} | trees={median_baseline_trees}"
    )

    add_xgb_result_row(
        dataset_name=fe_level,
        run_id=run_id,
        band_name=band_name,
        model_name="bce_xgb",
        split_pack=split_pack,
        t_ref=t_ref,
        t_min=t_min,
        t_max=t_max,
        test_nb=nb_bce,
        delta_vs_bce=0.0,
        hessian_mode="__BASELINE__",
        baseline_trees=median_baseline_trees,
        baseline_params=best_baseline_params,
    )

    # --------------------------------------------------
    # Local temperature scaling and local Platt scaling
    # Only for BCE-XGB
    # --------------------------------------------------
    calib = evaluate_bce_xgb_local_calibration(
        train_logits_base=train_logits_base,
        y_train=y_train,
        test_logits_base=test_logits_base,
        y_test=y_test,
        t_ref=float(t_ref),
        t_min=float(t_min),
        t_max=float(t_max),
    )

    nb_bce_temp = float(calib["bce_nb_temp_scaled"])
    nb_bce_platt = float(calib["bce_nb_platt_scaled"])

    add_xgb_result_row(
        dataset_name=fe_level,
        run_id=run_id,
        band_name=band_name,
        model_name="bce_xgb_local_temperature",
        split_pack=split_pack,
        t_ref=t_ref,
        t_min=t_min,
        t_max=t_max,
        test_nb=nb_bce_temp,
        delta_vs_bce=nb_bce_temp - nb_bce,
        hessian_mode="__BASELINE__",
        baseline_trees=median_baseline_trees,
        baseline_params=best_baseline_params,
        calibration_info=calib,
    )

    add_xgb_result_row(
        dataset_name=fe_level,
        run_id=run_id,
        band_name=band_name,
        model_name="bce_xgb_local_platt",
        split_pack=split_pack,
        t_ref=t_ref,
        t_min=t_min,
        t_max=t_max,
        test_nb=nb_bce_platt,
        delta_vs_bce=nb_bce_platt - nb_bce,
        hessian_mode="__BASELINE__",
        baseline_trees=median_baseline_trees,
        baseline_params=best_baseline_params,
        calibration_info=calib,
    )

    print(
        f"[CAL][{fe_level} | run {run_id} | {band_name}] "
        f"BCE-temp Δ={nb_bce_temp - nb_bce:+.6f} | "
        f"BCE-Platt Δ={nb_bce_platt - nb_bce:+.6f}"
    )

    # --------------------------------------------------
    # NB-XGB final refit and test evaluation
    # No calibration is applied to NB-XGB logits
    # --------------------------------------------------
    for h in hessian_modes:
        sel = selected_by_hessian[str(h)]

        best_cfg = dict(sel["best_nb_cfg"])
        best_cfg["_baseline_params_for_nb_stage"] = dict(
            sel["best_baseline_params"]
        )

        booster_nb = refit_nb_on_train_from_baseline_encoded(
            booster_baseline_train=booster_baseline_train,
            X_train_enc=X_train_enc,
            y_train=y_train,
            t_min=float(t_min),
            t_max=float(t_max),
            nb_cfg=best_cfg,
            hessian_mode=str(h),
            seed=int(seed),
        )

        test_logits_nb = booster_nb.predict(
            dtest,
            output_margin=True,
        )

        nb_snb = evaluate_nb_from_xgb_logits_np(
            test_logits_nb,
            y_test,
            thresh_min=float(t_min),
            thresh_max=float(t_max),
            num_points=int(TEST_RANGE_POINTS),
        )

        delta = float(nb_snb - nb_bce)

        print(
            f"[TEST][{fe_level} | run {run_id} | {band_name} | h={h}] "
            f"NB-XGB NB={nb_snb:+.6f} | Δ={delta:+.6f} | "
            f"base_trees={median_baseline_trees} + "
            f"added=({best_cfg.get('median_added_trees_it1', 0)}, "
            f"{best_cfg.get('median_added_trees_it4', 0)}, "
            f"{best_cfg.get('median_added_trees_it10', 0)})"
        )

        add_xgb_result_row(
            dataset_name=fe_level,
            run_id=run_id,
            band_name=band_name,
            model_name="nb_xgb",
            split_pack=split_pack,
            t_ref=t_ref,
            t_min=t_min,
            t_max=t_max,
            test_nb=nb_snb,
            delta_vs_bce=delta,
            hessian_mode=str(h),
            baseline_trees=median_baseline_trees,
            baseline_params=sel["best_baseline_params"],
            nb_added_trees=int(best_cfg.get("median_added_trees", 0)),
            nb_added_trees_it1=int(best_cfg.get("median_added_trees_it1", 0)),
            nb_added_trees_it4=int(best_cfg.get("median_added_trees_it4", 0)),
            nb_added_trees_it10=int(best_cfg.get("median_added_trees_it10", 0)),
            nb_anneal_schedule=best_cfg.get("anneal_inv_temps", NB_ANNEAL_INV_TEMPS),
            nb_params={
                k: v
                for k, v in best_cfg.items()
                if not k.startswith("_")
            },
        )


for fe_level, splits_list in DATASETS.items():
    print(f"\n==================== DATASET: {fe_level} ====================")

    for sp in splits_list:
        run_id = int(sp.run_id)

        X_train_enc = np.asarray(sp.X_train, dtype=np.float32)
        X_test_enc = np.asarray(sp.X_test, dtype=np.float32)

        y_train = np.asarray(sp.y_train, dtype=np.float32).reshape(-1)
        y_test = np.asarray(sp.y_test, dtype=np.float32).reshape(-1)

        band_specs = []

        for t_ref in (0.05, 0.10, 0.20):
            t_min, t_max = band_from_t_ref(t_ref)
            band_specs.append((f"t_{t_ref:.2f}", t_ref, t_min, t_max))

        for band_name, t_ref, t_min, t_max in band_specs:
            print(
                f"\n[{fe_level} | run {run_id}] "
                f"=== BAND: {band_name} "
                f"(t_ref={t_ref:.4f}, [{t_min:.4f}, {t_max:.4f}]) ==="
            )

            run_one_outer_rotation_xgb_with_bce_calibration(
                X_train_enc=X_train_enc,
                y_train=y_train,
                X_test_enc=X_test_enc,
                y_test=y_test,
                run_id=run_id,
                band_name=band_name,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                fe_level=fe_level,
                split_pack=sp,
                hessian_modes=HESSIAN_MODES,
                inner_cv_splits=5,
                seed=int(sp.seed),
            )


df_results_xgb = pd.DataFrame(xgb_results)

print("\nFinished.")
print("df_results_xgb shape:", df_results_xgb.shape)


==================== DATASET: mild ====================

[mild | run 0] === BAND: t_0.05 (t_ref=0.0500, [0.0250, 0.0750]) ===

[Inner CV][Baseline] 32 configs | metric=VAL logloss | folds=5
[Inner CV][Baseline 001/32] val_logloss=0.400706 | val_nb(info)=+0.125786 | median_trees=  62 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=1.0 | gamma=0.0 | subsample=1.0 | colsample=1.0
[Inner CV][Baseline 002/32] val_logloss=0.400663 | val_nb(info)=+0.125722 | median_trees=  82 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=1.0 | gamma=1.0 | subsample=1.0 | colsample=1.0
[Inner CV][Baseline 003/32] val_logloss=0.399798 | val_nb(info)=+0.125805 | median_trees=  88 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=5.0 | gamma=0.0 | subsample=1.0 | colsample=1.0
[Inner CV][Baseline 004/32] val_logloss=0.399316 | val_nb(info)=+0.125842 | median_trees=  88 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=5.0 | gamma=1.0 | subsample=1.0 | colsample=1.0
[

# Step 9 — Display XGBoost results

This step displays the XGBoost results in the same style as the LR notebook.

The baseline model is uncalibrated BCE-XGB. The comparison models are local temperature-scaled BCE-XGB, local Platt-scaled BCE-XGB, and annealed NB-XGB.

In [13]:


from scipy.stats import ttest_rel
import numpy as np
import pandas as pd

baseline_model = "bce_xgb"

calib_df = (
    df_results_xgb[
        df_results_xgb["hessian_mode"] == "__BASELINE__"
    ]
    .pivot_table(
        index=["dataset", "run_id", "band_name"],
        columns="model",
        values="test_nb",
        aggfunc="first",
    )
    .reset_index()
)

summary_rows = []

for (dataset_name, band_name), sub in calib_df.groupby(
    ["dataset", "band_name"]
):
    for model_name in [
        "bce_xgb_local_temperature",
        "bce_xgb_local_platt",
    ]:
        if model_name not in sub.columns:
            continue

        tmp = sub.dropna(
            subset=[baseline_model, model_name]
        ).copy()

        n_runs = len(tmp)

        if n_runs == 0:
            continue

        delta = tmp[model_name] - tmp[baseline_model]

        if n_runs >= 2:
            t_stat, p_value = ttest_rel(
                tmp[model_name],
                tmp[baseline_model],
            )
        else:
            t_stat = np.nan
            p_value = np.nan

        summary_rows.append({
            "dataset": dataset_name,
            "band_name": band_name,
            "model": model_name,
            "n_runs": int(n_runs),
            "mean_bce_xgb": float(tmp[baseline_model].mean()),
            "mean_model_nb": float(tmp[model_name].mean()),
            "mean_delta_vs_bce": float(delta.mean()),
            "sd_delta_vs_bce": (
                float(delta.std(ddof=1))
                if n_runs > 1 else np.nan
            ),
            "wins_vs_bce": int((delta > 0).sum()),
            "losses_vs_bce": int((delta < 0).sum()),
            "ties_vs_bce": int((delta == 0).sum()),
            "paired_t_vs_bce": float(t_stat),
            "paired_p_vs_bce": float(p_value),
            "significant_0.05": (
                bool(p_value < 0.05)
                if np.isfinite(p_value)
                else False
            ),
        })

summary_calib_xgb_df = pd.DataFrame(summary_rows)

print("\n======================================")
print("Calibration methods vs BCE-XGB")
print("======================================")
display(summary_calib_xgb_df)


Calibration methods vs BCE-XGB


,dataset,band_name,model,n_runs,mean_bce_xgb,mean_model_nb,mean_delta_vs_bce,sd_delta_vs_bce,wins_vs_bce,losses_vs_bce,ties_vs_bce,paired_t_vs_bce,paired_p_vs_bce,significant_0.05
0,mild,t_0.05,bce_xgb_local_temperature,5,0.126195,0.125146,-0.001049,0.001059,0,5,0,-2.215358,0.091081,False
1,mild,t_0.05,bce_xgb_local_platt,5,0.126195,0.125222,-0.000973,0.000934,1,4,0,-2.328566,0.080378,False
2,mild,t_0.10,bce_xgb_local_temperature,5,0.094785,0.092290,-0.002495,0.001865,0,5,0,-2.990482,0.040319,True
3,mild,t_0.10,bce_xgb_local_platt,5,0.094785,0.093177,-0.001608,0.001417,1,4,0,-2.536995,0.064185,False
4,mild,t_0.20,bce_xgb_local_temperature,5,0.046965,0.046852,-0.000113,0.001056,2,3,0,-0.238791,0.823003,False
5,mild,t_0.20,bce_xgb_local_platt,5,0.046965,0.046458,-0.000506,0.000287,0,5,0,-3.945712,0.016879,True
6,moderate,t_0.05,bce_xgb_local_temperature,5,0.126293,0.125499,-0.000794,0.001551,2,3,0,-1.145281,0.315959,False
7,moderate,t_0.05,bce_xgb_local_platt,5,0.126293,0.125551,-0.000742,0.001516,2,3,0,-1.094656,0.335160,False
8,moderate,t_0.10,bce_xgb_local_temperature,5,0.095191,0.093055,-0.002136,0.002057,0,5,0,-2.322139,0.080947,False
9,moderate,t_0.10,bce_xgb_local_platt,5,0.095191,0.093615,-0.001576,0.001462,0,5,0,-2.410273,0.073533,False


In [14]:
# ======================================================================
# NB-XGB significance tests per Hessian mode
# ======================================================================

baseline_rows = (
    df_results_xgb[
        df_results_xgb["model"] == "bce_xgb"
    ][
        ["dataset", "run_id", "band_name", "test_nb"]
    ]
    .rename(
        columns={
            "test_nb": "bce_test_nb"
        }
    )
)

nb_rows = (
    df_results_xgb[
        df_results_xgb["model"] == "nb_xgb"
    ][
        [
            "dataset",
            "run_id",
            "band_name",
            "hessian_mode",
            "test_nb",
        ]
    ]
    .rename(
        columns={
            "test_nb": "nb_test_nb"
        }
    )
)

nb_compare = nb_rows.merge(
    baseline_rows,
    on=["dataset", "run_id", "band_name"],
    how="left",
)

summary_rows = []

for (
    dataset_name,
    band_name,
    hessian_mode,
), sub in nb_compare.groupby(
    ["dataset", "band_name", "hessian_mode"]
):
    tmp = sub.dropna(
        subset=["bce_test_nb", "nb_test_nb"]
    ).copy()

    n_runs = len(tmp)

    if n_runs == 0:
        continue

    delta = (
        tmp["nb_test_nb"]
        - tmp["bce_test_nb"]
    )

    if n_runs >= 2:
        t_stat, p_value = ttest_rel(
            tmp["nb_test_nb"],
            tmp["bce_test_nb"],
        )
    else:
        t_stat = np.nan
        p_value = np.nan

    summary_rows.append({
        "dataset": dataset_name,
        "band_name": band_name,
        "hessian_mode": hessian_mode,
        "n_runs": int(n_runs),
        "mean_bce_xgb": float(
            tmp["bce_test_nb"].mean()
        ),
        "mean_nb_xgb": float(
            tmp["nb_test_nb"].mean()
        ),
        "mean_delta_vs_bce": float(
            delta.mean()
        ),
        "sd_delta_vs_bce": (
            float(delta.std(ddof=1))
            if n_runs > 1 else np.nan
        ),
        "wins_vs_bce": int(
            (delta > 0).sum()
        ),
        "losses_vs_bce": int(
            (delta < 0).sum()
        ),
        "ties_vs_bce": int(
            (delta == 0).sum()
        ),
        "paired_t_vs_bce": float(
            t_stat
        ),
        "paired_p_vs_bce": float(
            p_value
        ),
        "significant_0.05": (
            bool(p_value < 0.05)
            if np.isfinite(p_value)
            else False
        ),
    })

summary_nb_xgb_df = pd.DataFrame(summary_rows)

print("\n======================================")
print("NB-XGB versus BCE-XGB")
print("======================================")
display(summary_nb_xgb_df)


NB-XGB versus BCE-XGB


,dataset,band_name,hessian_mode,n_runs,mean_bce_xgb,mean_nb_xgb,mean_delta_vs_bce,sd_delta_vs_bce,wins_vs_bce,losses_vs_bce,ties_vs_bce,paired_t_vs_bce,paired_p_vs_bce,significant_0.05
0,mild,t_0.05,absw,5,0.126195,0.125567,-0.000628,0.001290,1,4,0,-1.088443,0.337588,False
1,mild,t_0.05,fixed025,5,0.126195,0.125564,-0.000631,0.001718,3,2,0,-0.821617,0.457444,False
2,mild,t_0.05,true,5,0.126195,0.124200,-0.001995,0.001808,0,4,1,-2.467885,0.069104,False
3,mild,t_0.10,absw,5,0.094785,0.094297,-0.000487,0.002535,2,2,1,-0.429866,0.689438,False
4,mild,t_0.10,fixed025,5,0.094785,0.094307,-0.000478,0.001065,3,2,0,-1.002647,0.372766,False
5,mild,t_0.10,true,5,0.094785,0.095120,0.000336,0.001159,2,2,1,0.647684,0.552487,False
6,mild,t_0.20,absw,5,0.046965,0.044867,-0.002098,0.002582,1,4,0,-1.816376,0.143480,False
7,mild,t_0.20,fixed025,5,0.046965,0.046036,-0.000929,0.001663,2,3,0,-1.248818,0.279829,False
8,mild,t_0.20,true,5,0.046965,0.046121,-0.000844,0.001336,1,3,1,-1.411584,0.230917,False
9,moderate,t_0.05,absw,5,0.126293,0.125540,-0.000753,0.002141,2,3,0,-0.786600,0.475502,False
